# Urban Pulse NYC - Analyst Tasks 12 to 15 (distribution, geography, closure, volume + delay)
Run the cells **in order, top to bottom**.

**Rules used here (same as Tasks 7-11):** a *valid resolution duration* is a non-blank `resolution_hours` (114,660 rows); group medians are shown only when a group has **at least 100 valid durations**; outliers are flagged and kept, never deleted.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

PROJECT = "C:/Users/skmah/urban_pulse"
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)
print("Working folder:", os.getcwd())

os.makedirs("figures", exist_ok=True)
os.makedirs("sql", exist_ok=True)

engine = create_engine("mysql+pymysql://root:@localhost:3306/urban_pulse_db")
pd.options.display.width = 150
pd.options.display.float_format = "{:,.2f}".format

MIN_N = 100
QUERIES = {}

def task(num, title, sql):
    """Run a SQL query, remember it, and return the result as a DataFrame."""
    QUERIES[num] = (title, sql.strip())
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def save_fig(num, name):
    path = f"figures/task{num:02d}_{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved chart:", path)

with engine.connect() as conn:
    TOTAL = int(pd.read_sql(text("SELECT COUNT(*) AS n FROM service_requests"), conn).loc[0, "n"])
    VALID = int(pd.read_sql(text("SELECT COUNT(resolution_hours) AS n FROM service_requests"), conn).loc[0, "n"])
print(f"Connected. Rows: {TOTAL:,} | valid resolution durations: {VALID:,}")

## Task 12 - Resolution-time skewness and distribution
**Question:** What does the distribution of `resolution_hours` look like, and how skewed is it? (Histograms and boxplot, as required by the project document.)

In [ ]:
res = task(12, "Valid resolution durations (values used for distribution)",
    """SELECT resolution_hours
       FROM service_requests
       WHERE resolution_hours IS NOT NULL""")["resolution_hours"]

skew, kurt = res.skew(), res.kurt()
log_res = np.log1p(res)                       # log(1 + hours): a common fix for right-skewed data
summary12 = pd.Series({
    "valid durations (n)": len(res),
    "mean (hours)": res.mean(),
    "median (hours)": res.median(),
    "mean / median ratio": res.mean() / res.median(),
    "skewness": skew,
    "excess kurtosis": kurt,
    "skewness after log(1+x)": log_res.skew(),
    "share over 72 h (%)": 100 * (res > 72).mean(),
    "durations of exactly 0 h": int((res == 0).sum()),
}, name="value")
display(summary12.to_frame())
print("Skewness > 1 means strongly right-skewed: a few very long cases pull the mean far above the median.")

In [ ]:
df12b = task("12b", "Resolution time in duration buckets",
    """SELECT CASE
                WHEN resolution_hours < 1   THEN '1) under 1 hour'
                WHEN resolution_hours < 24  THEN '2) 1 to 24 hours'
                WHEN resolution_hours < 72  THEN '3) 24 to 72 hours'
                WHEN resolution_hours < 168 THEN '4) 3 to 7 days'
                ELSE                             '5) over 7 days'
              END AS duration_bucket,
              COUNT(*) AS requests,
              ROUND(100 * COUNT(*) / (SELECT COUNT(resolution_hours) FROM service_requests), 2) AS pct_of_valid
       FROM service_requests
       WHERE resolution_hours IS NOT NULL
       GROUP BY duration_bucket
       ORDER BY duration_bucket""")
display(df12b)
print("Over 72 h = buckets 4 + 5 =", round(df12b["pct_of_valid"].iloc[-2:].sum(), 2), "% (the dashboard KPI)")

p95 = res.quantile(0.95)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].hist(res, bins=60, color="steelblue")
axes[0, 0].set_title("All durations (heavily right-skewed)")
axes[0, 0].set_xlabel("Resolution hours"); axes[0, 0].set_ylabel("Requests")
axes[0, 1].hist(res[res <= p95], bins=60, color="steelblue")
axes[0, 1].set_title(f"Up to the 95th percentile ({p95:,.0f} h, zoomed in)")
axes[0, 1].set_xlabel("Resolution hours")
axes[1, 0].boxplot(res[res > 0], showfliers=True, flierprops={"markersize": 2})
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Boxplot (log scale, zero-hour values not shown)")
axes[1, 0].set_ylabel("Resolution hours (log)")
axes[1, 0].set_xticks([])
axes[1, 1].hist(log_res, bins=60, color="seagreen")
axes[1, 1].set_title("After log(1 + hours): much less skewed")
axes[1, 1].set_xlabel("log(1 + resolution hours)")
fig.suptitle("Task 12: Distribution of resolution time")
plt.tight_layout()
save_fig(12, "resolution_distribution")
plt.show()

## Task 13 - Geographic complaint concentration
**Question:** Where are requests concentrated? Uses coordinates that are present and inside the NYC area (`coord_missing_flag = 0` and `coord_out_of_nyc_flag = 0`), plus ZIP-code and borough summaries.

In [ ]:
df13a = task("13a", "Coordinate usability",
    """SELECT COUNT(*) AS total_requests,
              CAST(SUM(coord_missing_flag) AS SIGNED)     AS missing_coordinates,
              CAST(SUM(coord_out_of_nyc_flag) AS SIGNED)  AS outside_nyc_box,
              CAST(SUM(coord_missing_flag = 0 AND coord_out_of_nyc_flag = 0) AS SIGNED) AS usable_coordinates
       FROM service_requests""")
df13a["usable_pct"] = 100 * df13a["usable_coordinates"] / df13a["total_requests"]
display(df13a)

df13b = task("13b", "Top 10 ZIP codes by request count",
    """SELECT incident_zip,
              COUNT(*) AS requests,
              ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM service_requests
                                      WHERE incident_zip IS NOT NULL AND zip_invalid_flag = 0), 2) AS pct_of_valid_zip
       FROM service_requests
       WHERE incident_zip IS NOT NULL AND zip_invalid_flag = 0
       GROUP BY incident_zip
       ORDER BY requests DESC
       LIMIT 10""")
print("Top 10 ZIP codes (n = rows with a valid ZIP):")
display(df13b)

df13c = task("13c", "Top 3 complaint types in each borough",
    """WITH counts AS (
         SELECT borough, complaint_type, COUNT(*) AS requests
         FROM service_requests
         WHERE borough <> 'UNSPECIFIED'
         GROUP BY borough, complaint_type
       ),
       ranked AS (
         SELECT borough, complaint_type, requests,
                ROW_NUMBER() OVER (PARTITION BY borough ORDER BY requests DESC) AS rank_in_borough,
                SUM(requests) OVER (PARTITION BY borough)                       AS borough_total
         FROM counts
       )
       SELECT borough, rank_in_borough, complaint_type, requests,
              ROUND(100 * requests / borough_total, 1) AS pct_of_borough
       FROM ranked
       WHERE rank_in_borough <= 3
       ORDER BY borough, rank_in_borough""")
print("Top 3 complaint types per borough:")
display(df13c)

In [ ]:
pts = task("13d", "Usable coordinates for the density map",
    """SELECT latitude, longitude
       FROM service_requests
       WHERE coord_missing_flag = 0 AND coord_out_of_nyc_flag = 0
         AND latitude IS NOT NULL AND longitude IS NOT NULL""")
print("Points plotted (n):", f"{len(pts):,}", "of", f"{TOTAL:,}", "requests")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.3, 1]})
hb = axes[0].hexbin(pts["longitude"], pts["latitude"], gridsize=70, bins="log", cmap="YlOrRd", mincnt=1)
axes[0].set_aspect(1 / np.cos(np.radians(40.7)))      # keeps the map from looking stretched
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
axes[0].set_title("Request density (darker = more requests, log colour scale)")
fig.colorbar(hb, ax=axes[0], label="Requests per map cell (log scale)")
z = df13b.iloc[::-1]
axes[1].barh(z["incident_zip"].astype(str), z["requests"], color="firebrick")
axes[1].set_xlabel("Requests (3% sample)")
axes[1].set_title("Top 10 ZIP codes")
fig.suptitle("Task 13: Geographic concentration of requests")
plt.tight_layout()
save_fig(13, "geographic_concentration")
plt.show()

## Task 14 - Recorded closure-rate comparison
**Question:** What share of requests has a recorded closed date (`is_closed = 1`), and how does it differ by borough, agency and creation month?  
Closure rate = recorded closure only. It is **not** a measure of satisfaction or verified resolution.

In [ ]:
overall = task("14-overall", "Overall recorded closure rate",
    """SELECT COUNT(*) AS requests,
              CAST(SUM(is_closed) AS SIGNED) AS closed,
              ROUND(100 * SUM(is_closed) / COUNT(*), 2) AS closure_rate_pct
       FROM service_requests""")
display(overall)
OVERALL_RATE = float(overall.loc[0, "closure_rate_pct"])

df14a = task("14a", "Closure rate by borough",
    """SELECT borough,
              COUNT(*) AS requests,
              CAST(SUM(is_closed) AS SIGNED) AS closed,
              ROUND(100 * SUM(is_closed) / COUNT(*), 2) AS closure_rate_pct
       FROM service_requests
       GROUP BY borough
       ORDER BY closure_rate_pct DESC""")
print("By borough:"); display(df14a)

df14b = task("14b", "Closure rate by agency",
    """SELECT agency,
              COUNT(*) AS requests,
              CAST(SUM(is_closed) AS SIGNED) AS closed,
              ROUND(100 * SUM(is_closed) / COUNT(*), 2) AS closure_rate_pct
       FROM service_requests
       GROUP BY agency
       ORDER BY closure_rate_pct DESC""")
print("By agency (agencies with fewer than 100 requests are listed but not charted):"); display(df14b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
b = df14a[df14a["requests"] >= MIN_N].sort_values("closure_rate_pct")
axes[0].barh(b["borough"], b["closure_rate_pct"], color="darkorange")
axes[0].set_title("By borough")
a = df14b[df14b["requests"] >= MIN_N].sort_values("closure_rate_pct")
axes[1].barh(a["agency"], a["closure_rate_pct"], color="seagreen")
axes[1].set_title(f"By agency (n >= {MIN_N})")
for ax in axes:
    ax.axvline(OVERALL_RATE, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Recorded closure rate (%)")
    ax.set_xlim(max(0, min(b["closure_rate_pct"].min(), a["closure_rate_pct"].min()) - 5), 100.5)
fig.suptitle(f"Task 14: Recorded closure rate (dashed line = overall {OVERALL_RATE}%)")
plt.tight_layout()
save_fig(14, "closure_rate_borough_agency")
plt.show()

In [ ]:
df14c = task("14c", "Closure rate by creation month",
    """SELECT created_ym,
              COUNT(*) AS requests,
              ROUND(100 * SUM(is_closed) / COUNT(*), 2) AS closure_rate_pct
       FROM service_requests
       GROUP BY created_ym
       ORDER BY created_ym""")
display(df14c)

plt.figure(figsize=(10, 4.5))
plt.plot(df14c["created_ym"], df14c["closure_rate_pct"], marker="o", color="purple")
plt.ylim(max(0, df14c["closure_rate_pct"].min() - 5), 100.5)
plt.ylabel("Recorded closure rate (%)")
plt.title("Task 14: Closure rate by month the request was created")
plt.xticks(rotation=45); plt.grid(alpha=0.3)
save_fig(14, "closure_rate_by_month")
plt.show()
print("Why this matters: recent requests have had less time to be closed, so a lower rate in the latest months may just reflect timing.")

## Task 15 - Categories with both high request volume and long median resolution duration
**Question:** Which complaint types are both frequent and slow to reach recorded closure?  
**Project-defined cut-offs (state this in your documentation):** among categories with at least 100 valid durations, *high volume* = requests at or above the 75th percentile of category volume; *long median* = median hours at or above the 75th percentile of category medians. These are analytical cut-offs, not official standards.

In [ ]:
t15 = task(15, "Complaint types: request volume and median resolution hours",
    """WITH ranked AS (
         SELECT complaint_type,
                resolution_hours,
                ROW_NUMBER() OVER (PARTITION BY complaint_type ORDER BY resolution_hours) AS rn,
                COUNT(*)     OVER (PARTITION BY complaint_type)                           AS n
         FROM service_requests
         WHERE resolution_hours IS NOT NULL
       ),
       med AS (
         SELECT complaint_type,
                MAX(n) AS valid_durations,
                ROUND(AVG(CASE WHEN rn IN (FLOOR((n + 1) / 2), CEIL((n + 1) / 2))
                               THEN resolution_hours END), 2) AS median_hours,
                ROUND(AVG(resolution_hours), 2) AS mean_hours
         FROM ranked
         GROUP BY complaint_type
       ),
       vol AS (
         SELECT complaint_type, COUNT(*) AS requests
         FROM service_requests
         GROUP BY complaint_type
       )
       SELECT v.complaint_type, v.requests, m.valid_durations, m.median_hours, m.mean_hours
       FROM vol v
       JOIN med m ON m.complaint_type = v.complaint_type""")

elig = t15[t15["valid_durations"] >= MIN_N].copy()
vol_cut = elig["requests"].quantile(0.75)
med_cut = elig["median_hours"].quantile(0.75)
elig["high_volume"] = elig["requests"] >= vol_cut
elig["long_median"] = elig["median_hours"] >= med_cut
both = elig[elig["high_volume"] & elig["long_median"]].sort_values("requests", ascending=False)

print(f"Categories with at least {MIN_N} valid durations: {len(elig)} (of {len(t15)})")
print(f"High-volume cut-off: >= {vol_cut:,.0f} requests | Long-median cut-off: >= {med_cut:,.1f} hours")
both = both.assign(pct_of_all_requests=100 * both["requests"] / TOTAL)
print("\nCategories that are BOTH high volume AND long median:")
display(both[["complaint_type", "requests", "pct_of_all_requests", "valid_durations", "median_hours", "mean_hours"]])
print("These categories together =", f"{int(both['requests'].sum()):,}", "requests (",
      round(100 * both["requests"].sum() / TOTAL, 1), "% of all requests )")

plt.figure(figsize=(11, 7))
plt.scatter(elig["requests"], elig["median_hours"].clip(lower=0.1), color="lightgray", edgecolor="gray", label="Other categories")
plt.scatter(both["requests"], both["median_hours"].clip(lower=0.1), color="crimson", s=60, label="High volume + long median")
for _, r in both.iterrows():
    plt.annotate(r["complaint_type"], (r["requests"], max(r["median_hours"], 0.1)),
                 xytext=(5, 4), textcoords="offset points", fontsize=8)
plt.axvline(vol_cut, color="black", linestyle="--", linewidth=1)
plt.axhline(med_cut, color="black", linestyle="--", linewidth=1)
plt.axhline(72, color="gray", linestyle=":", linewidth=1)
plt.xscale("log"); plt.yscale("log")
plt.text(elig["requests"].min(), 72, " illustrative 72 h", va="bottom", fontsize=8, color="gray")
plt.xlabel("Requests (3% sample, log scale)")
plt.ylabel("Median resolution hours (log scale)")
plt.title("Task 15: Request volume vs median resolution time")
plt.legend(loc="lower left")
save_fig(15, "volume_vs_median_resolution")
plt.show()

## Save all queries from this notebook
Creates `sql/04_tasks_12_15.sql`.

In [ ]:
with open("sql/04_tasks_12_15.sql", "w", encoding="utf-8") as f:
    f.write("-- Urban Pulse NYC: analyst tasks 12-15 (run against urban_pulse_db)\n")
    f.write("-- Note: medians use window functions because MySQL has no MEDIAN().\n\n")
    for num, (title, sql) in QUERIES.items():
        f.write(f"-- Task {num}: {title}\n{sql};\n\n")
print("Saved sql/04_tasks_12_15.sql with", len(QUERIES), "queries")